# GSSS_011 - Build a mini ChatGPT: one model, many tools

Labs so far built the pieces separately. This one **wires them into a single app**:

| ChatGPT feature | How we do it (all free) |
|---|---|
| chat history sidebar | SQLite - two tables (Lab 3) |
| remembers you across chats | a `memories` table the model reconciles (Lab 4), recalled by TF-IDF - **no model download** |
| runs code / analyses files | a `run_python` tool - `exec` in a pandas + matplotlib namespace |
| searches the web | `web_search` tool -> DuckDuckGo, falling back to Wikipedia on rate limits (no key) |
| makes images | `generate_image` tool -> Pollinations (no key) |
| speaks | `speak` tool -> gTTS (no key) |
| voice input | Groq Whisper transcribes the mic before the model sees it |

**One `create_agent` holds the tools and decides what to call.** The model is Groq
`qwen/qwen3.8-27b` (reliable at tool-calling), falling back to `gpt-oss-120b` / OpenRouter.
The point is the *architecture*, not answer quality - every model here is small and free.


## Step 0 - Install

In [1]:
%pip install -q langchain langchain-groq langchain-openai \
    scikit-learn ddgs gradio groq gtts requests pillow pandas openpyxl matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 41.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 1.16.1 which is incompatible.


## Step 1 - Keys + models

`GROQ_API_KEY` is required (chat **and** Whisper). `OPENROUTER_API_KEY` is an optional
fallback. Keys are pasted at runtime, never saved into the notebook.

In [2]:
import os, re, json, time, io, contextlib

def ask(name, required=True):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            print(f"{name}: from Colab secret"); return v
    except Exception:
        pass
    if os.getenv(name):
        print(f"{name}: from environment"); return os.environ[name]
    from getpass import getpass
    tail = "" if required else "  (optional - press Enter to skip)"
    return getpass(f"Paste {name}{tail}: ").strip()

GROQ_API_KEY       = ask("GROQ_API_KEY")
OPENROUTER_API_KEY = ask("OPENROUTER_API_KEY", required=False)
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

# agent model: qwen is dependable at tool-calling; the rest are fallbacks
AGENT_MODELS = [
    ChatGroq(model="qwen/qwen3.8-27b", api_key=GROQ_API_KEY, temperature=0,
             max_retries=2, request_timeout=45),
    ChatGroq(model="openai/gpt-oss-120b", api_key=GROQ_API_KEY, temperature=0,
             max_retries=1, request_timeout=45),
]
if OPENROUTER_API_KEY:
    AGENT_MODELS.append(ChatOpenAI(model="minimax/minimax-m2.7:free",
        base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY,
        temperature=0, max_retries=1, request_timeout=45))

def _name(m):
    return getattr(m, "model_name", getattr(m, "model", "?"))

def plain_llm(prompt):
    """One plain completion (used for memory reconcile + titles). Tries each model, then waits and retries."""
    err = None
    for wait in (0, 12, 30):
        if wait:
            time.sleep(wait)
        for m in AGENT_MODELS:
            try:
                return m.invoke(prompt).content
            except Exception as e:
                err = e
    raise err

print("agent models:", [_name(m) for m in AGENT_MODELS])
print(plain_llm("Reply with one word: ready"))

GROQ_API_KEY: from Colab secret
OPENROUTER_API_KEY: from Colab secret
agent models: ['qwen/qwen3.8-27b', 'openai/gpt-oss-120b', 'minimax/minimax-m2.7:free']
Ready


## Step 2 - Chat history (SQLite, from Lab 3)

`conversations` = the sidebar list, `messages` = the transcript. Every turn is written to
disk, so switching chats or restarting loses nothing.

In [3]:
import sqlite3, datetime

DB = "mini_chatgpt.db"

def db():
    con = sqlite3.connect(DB)
    con.execute("PRAGMA foreign_keys = ON")
    return con

with db() as con:
    con.executescript("""
        CREATE TABLE IF NOT EXISTS conversations (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL DEFAULT 'New chat',
            created_at TEXT NOT NULL);
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            conversation_id INTEGER NOT NULL REFERENCES conversations(id) ON DELETE CASCADE,
            role TEXT NOT NULL, content TEXT NOT NULL, created_at TEXT NOT NULL);
    """)

def _now():
    return datetime.datetime.now().isoformat(timespec="seconds")

def new_conversation(title="New chat"):
    with db() as con:
        return con.execute("INSERT INTO conversations(title, created_at) VALUES (?, ?)",
                           (title, _now())).lastrowid

def list_conversations():
    with db() as con:
        return con.execute("SELECT id, title FROM conversations ORDER BY id DESC").fetchall()

def get_messages(cid):
    with db() as con:
        return con.execute("SELECT role, content FROM messages WHERE conversation_id=? ORDER BY id",
                           (cid,)).fetchall()

def add_message(cid, role, content):
    with db() as con:
        con.execute("INSERT INTO messages(conversation_id, role, content, created_at) VALUES (?,?,?,?)",
                    (cid, role, content, _now()))

def rename_conversation(cid, title):
    with db() as con:
        con.execute("UPDATE conversations SET title=? WHERE id=?", (title, cid))

def delete_conversation(cid):
    with db() as con:
        con.execute("DELETE FROM conversations WHERE id=?", (cid,))

def auto_title(first_msg):
    raw = plain_llm("3-5 word title for a chat starting with this message. Title only, one line.\n\n"
                    + first_msg).strip()
    return raw.splitlines()[0].strip().strip('"').lstrip("#").strip()[:50] or "New chat"

print("chat DB ready:", DB)

chat DB ready: mini_chatgpt.db


## Step 3 - Memory (from Lab 4, made dependency-free)

A `memories` table the model **reconciles** (add / update / delete) after every turn. On
each new message, `recall()` returns only the memories similar to it; **preferences always
apply**, facts/projects are recalled when relevant.

Lab 4 used a downloaded MiniLM embedding model. Here we use **TF-IDF cosine** instead -
pure `scikit-learn`, nothing to download, no rate limits. It matches on shared words rather
than deeper meaning, which is enough at this scale; swap MiniLM back in if you want
meaning-based recall.

In [4]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

KINDS = ["preference", "fact", "project"]

with db() as con:
    con.execute("""CREATE TABLE IF NOT EXISTS memories (
        id INTEGER PRIMARY KEY AUTOINCREMENT, kind TEXT NOT NULL, content TEXT NOT NULL,
        created_at TEXT NOT NULL)""")

def all_mem():
    with db() as con:
        return con.execute("SELECT id, kind, content FROM memories ORDER BY id").fetchall()

def _sim(query, texts):
    """TF-IDF cosine of query vs each text. Refit every call - trivial for a few dozen rows."""
    if not texts:
        return np.array([])
    v = TfidfVectorizer().fit(texts + [query])
    return cosine_similarity(v.transform([query]), v.transform(texts))[0]

def add_mem(kind, content):
    kind = kind if kind in KINDS else "fact"
    rows = all_mem()
    if rows:
        sims = _sim(content, [c for _, _, c in rows])
        if sims.size and sims.max() > 0.75:            # near-duplicate already stored
            return rows[int(sims.argmax())][0]
    with db() as con:
        return con.execute("INSERT INTO memories(kind, content, created_at) VALUES (?,?,?)",
                           (kind, content, _now())).lastrowid

def update_mem(i, content):
    with db() as con:
        con.execute("UPDATE memories SET content=?, created_at=? WHERE id=?", (content, _now(), i))

def delete_mem(i):
    with db() as con:
        con.execute("DELETE FROM memories WHERE id=?", (i,))

def recall(query, k=4, threshold=0.08):
    rows = all_mem()
    if not rows:
        return []
    sims = _sim(query, [c for _, _, c in rows])
    ranked = sorted(zip(rows, sims), key=lambda x: x[1], reverse=True)
    return [(i, kd, c) for (i, kd, c), s in ranked[:k] if s > threshold]

RECONCILE = """You maintain long-term memory about ONE user.
CURRENT MEMORIES:
{cur}
The user just wrote: "{msg}"
Output ONLY a JSON list of operations (or [] for small talk / one-off questions):
  {{"op":"add","kind":"preference|fact|project","content":"..."}}
  {{"op":"update","id":<id>,"content":"..."}}
  {{"op":"delete","id":<id>}}"""

def reconcile_memory(msg, verbose=False):
    cur = "\n".join(f"#{i} [{k}] {c}" for i, k, c in all_mem()) or "(none)"
    raw = plain_llm(RECONCILE.format(cur=cur, msg=msg))
    m = re.search(r"\[.*\]", raw, re.S)
    try:
        ops = json.loads(m.group(0)) if m else []
    except Exception:
        ops = []
    touched = []
    for op in ops:
        try:
            if op["op"] == "add":
                touched.append(add_mem(op.get("kind", "fact"), op["content"].strip()))
            elif op["op"] == "update":
                update_mem(int(op["id"]), op["content"].strip()); touched.append(int(op["id"]))
            elif op["op"] == "delete":
                delete_mem(int(op["id"])); touched.append(int(op["id"]))
        except Exception:
            pass
    if verbose:
        print("  memory ops:", ops or "(none)")
    return touched

def memory_block(recalled):
    prefs = [c for i, k, c in all_mem() if k == "preference"]
    facts = [f"({k}) {c}" for i, k, c in recalled if k != "preference"]
    out = []
    if prefs:
        out.append("User preferences (ALWAYS follow):\n" + "\n".join(f"- {p}" for p in prefs))
    if facts:
        out.append("Relevant about this user:\n" + "\n".join(f"- {f}" for f in facts))
    return "\n\n".join(out)

print("memory ready")

memory ready


## Step 4 - The tools

Four tools. Each writes any file it makes into `MEDIA`, which the app renders after the
turn. Docstrings are what the model reads to choose - keep them sharp.

- `run_python` - math, data analysis, file creation, **and inspecting an uploaded file**
  (its path arrives in the `UPLOADED` variable)
- `web_search` - DuckDuckGo; **falls back to Wikipedia** when DDG rate-limits (no key)
- `generate_image` - a picture from a description (Pollinations, no key)
- `speak` - text to an MP3 (gTTS, no key)

In [5]:
from langchain_core.tools import tool
import requests, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

WORK = os.path.abspath("workspace"); os.makedirs(WORK, exist_ok=True)
MEDIA = []        # [(kind, path)] produced during the current turn
UPLOADED = [None] # path of the file the user attached, if any

@tool
def run_python(code: str) -> str:
    """Run Python 3 and return its stdout. Use for arithmetic, data analysis, and creating or
    inspecting files. pandas / numpy / matplotlib are imported. If the user attached a file,
    its path is in the variable UPLOADED (e.g. pd.read_csv(UPLOADED)). Save plots with
    plt.savefig('name.png')."""
    before = set(os.listdir(WORK))
    ns = {"pd": __import__("pandas"), "np": np, "plt": plt, "UPLOADED": UPLOADED[0], "os": os}
    buf = io.StringIO()
    cwd = os.getcwd(); os.chdir(WORK)
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, ns)
        err = ""
    except Exception as e:
        err = f"\nError: {type(e).__name__}: {e}"
    finally:
        os.chdir(cwd)
    for i, num in enumerate(plt.get_fignums()):
        f = os.path.join(WORK, f"plot_{int(time.time())}_{i}.png")
        plt.figure(num).savefig(f, bbox_inches="tight"); MEDIA.append(("image", f))
    plt.close("all")
    made = sorted(set(os.listdir(WORK)) - before)
    for fn in made:
        if not fn.startswith("plot_"):
            MEDIA.append(("file", os.path.join(WORK, fn)))
    out = buf.getvalue().strip() or "(no printed output)"
    return out + (f"\n[files created: {', '.join(made)}]" if made else "") + err

@tool
def web_search(query: str) -> str:
    """Search the web for current or factual info (news, prices, people, products, definitions).
    Returns short snippets with links. Use whenever the answer might be newer than your training data."""
    try:                                                   # primary: DuckDuckGo
        from ddgs import DDGS
        hits = list(DDGS().text(query, max_results=5))
        if hits:
            return "\n".join(f"- {h['title']}: {h['body'][:200]}  <{h['href']}>" for h in hits)
    except Exception as e:
        print("  [web_search] DDG failed ->", str(e)[:80], "- falling back to Wikipedia")
    try:                                                   # fallback: Wikipedia REST
        ua = {"User-Agent": "GSSS-Bot/1.0"}
        srch = requests.get("https://en.wikipedia.org/w/api.php", headers=ua, timeout=15, params={
            "action": "query", "list": "search", "srsearch": query,
            "format": "json", "srlimit": 3}).json()
        out = []
        for r in srch["query"]["search"]:
            t = r["title"]
            summ = requests.get("https://en.wikipedia.org/api/rest_v1/page/summary/"
                                + t.replace(" ", "_"), headers=ua, timeout=15).json()
            out.append(f"- {t}: {summ.get('extract', '')[:250]}")
        return "(DuckDuckGo unavailable - Wikipedia results)\n" + "\n".join(out) if out else "no results"
    except Exception as e:
        return f"search unavailable: {e}"

@tool
def generate_image(prompt: str) -> str:
    """Create an image from a text description. Use when the user wants a picture, drawing, logo or diagram."""
    try:
        url = "https://image.pollinations.ai/prompt/" + requests.utils.quote(prompt) + \
              "?width=768&height=512&nologo=true"
        p = os.path.join(WORK, f"img_{int(time.time())}.png")
        open(p, "wb").write(requests.get(url, timeout=90).content)
        MEDIA.append(("image", p))
        return f"Image created for: {prompt}"
    except Exception as e:
        return f"image error: {e}"

@tool
def speak(text: str) -> str:
    """Convert text to a spoken MP3. Use when the user asks to hear something read aloud."""
    try:
        from gtts import gTTS
        p = os.path.join(WORK, f"speak_{int(time.time())}.mp3")
        gTTS(text=text[:800]).save(p)
        MEDIA.append(("audio", p))
        return f"Spoken audio ready ({len(text)} chars)."
    except Exception as e:
        return f"tts error: {e}"

TOOLS = [run_python, web_search, generate_image, speak]
print("tools:", [t.name for t in TOOLS])

tools: ['run_python', 'web_search', 'generate_image', 'speak']


## Step 5 - Voice input: Groq Whisper

STT is **not** a tool the model calls - it runs first. Mic audio -> `transcribe()` -> the
text becomes the user's message, exactly as if they had typed it.

In [6]:
from groq import Groq
_gc = Groq(api_key=GROQ_API_KEY)

def transcribe(audio_path):
    if not audio_path or not os.path.exists(audio_path):
        return ""
    try:
        with open(audio_path, "rb") as f:
            return _gc.audio.transcriptions.create(model="whisper-large-v3", file=f).text.strip()
    except Exception as e:
        print("  [transcribe] failed:", str(e)[:100])
        return ""      # degrade gracefully - the user can just type instead

print("Whisper ready (record something in the app to try it)")

Whisper ready (record something in the app to try it)


## Step 6 - One turn: recall -> agent -> persist -> reconcile

`respond()` is the whole loop. The agent (a fresh `create_agent` each turn, so the current
memory is in its system prompt) decides whether to answer directly or call a tool.

In [7]:
from langchain.agents import create_agent

SYSTEM = (
    "You are a helpful assistant with tools:\n"
    "- run_python: math, data analysis, reading/creating files\n"
    "- web_search: current or factual info that may be newer than your training data\n"
    "- generate_image: any picture, drawing, logo or diagram the user wants\n"
    "- speak: turn text into spoken audio\n"
    "RULES: if the user asks to hear / say / read something aloud or 'out loud', you MUST "
    "call speak - never claim you spoke without calling it. If they want a picture, you MUST "
    "call generate_image. For anything time-sensitive or that you are unsure about, call "
    "web_search. Otherwise answer directly. Be concise.")

def run_agent(agents, messages, max_steps=8):
    err = None
    for wait in (0, 12, 30):
        if wait:
            time.sleep(wait)
        for ag in agents:
            try:
                out = ag.invoke({"messages": messages}, {"recursion_limit": max_steps * 2 + 3})
                trace = [tc["name"] for m in out["messages"]
                         for tc in (getattr(m, "tool_calls", None) or [])]
                return out["messages"][-1].content, trace
            except Exception as e:
                err = e
    raise err

def respond(conversation_id, user_text, verbose=True):
    MEDIA.clear()
    recalled = recall(user_text)
    sys = SYSTEM + ("\n\n" + memory_block(recalled) if memory_block(recalled) else "")
    if UPLOADED[0]:
        sys += f"\n\nThe user attached a file at: {UPLOADED[0]}  (use run_python to read it)."
    history = get_messages(conversation_id)
    agents = [create_agent(m, TOOLS, system_prompt=sys) for m in AGENT_MODELS]
    msgs = [{"role": r, "content": c} for r, c in history] + [{"role": "user", "content": user_text}]

    reply, trace = run_agent(agents, msgs)

    # safety net: small models sometimes SAY they spoke without calling the tool.
    # if the user clearly asked for audio and it was skipped, produce it deterministically.
    if "speak" not in trace and re.search(r"\b(out ?loud|aloud|read (it|this|that) (aloud|out)|"
                                          r"say .*(aloud|out ?loud))\b", user_text, re.I):
        speak.invoke({"text": reply})
        trace.append("speak")

    add_message(conversation_id, "user", user_text)
    add_message(conversation_id, "assistant", reply)
    if not history:
        rename_conversation(conversation_id, auto_title(user_text))
    touched = reconcile_memory(user_text)

    if verbose:
        print("TOOLS:", trace or "(none)")
        print("MEDIA:", [k for k, _ in MEDIA] or "(none)")
        print("MEMORY changed:", touched or "(none)")
        print("\nREPLY:\n" + reply[:1500])
    return reply, trace, list(MEDIA), recalled, touched

## Step 7 - Quick checks (one capability each)

A short pause between calls keeps us under the free-tier rate limit.

In [8]:
cid = new_conversation()

print("=== 1. plain answer (no tool) ==="); respond(cid, "In one sentence, what is an API?")
time.sleep(3)
print("\n=== 2. run_python ==="); respond(cid, "Use python: what is 17*23 and the mean of [4,8,15,16,23,42]?")
time.sleep(3)
print("\n=== 3. web_search ==="); respond(cid, "Search the web: who is the current CEO of OpenAI?")
time.sleep(3)
print("\n=== 4. generate_image ==="); respond(cid, "Draw a cartoon cat riding a bicycle.")
time.sleep(3)
print("\n=== 5. speak ==="); respond(cid, "Say 'hello from the mini ChatGPT' out loud.")
time.sleep(3)
print("\n=== 6. memory ==="); respond(cid, "Remember that I'm learning Spanish and I prefer short bullet-point answers.")
print("\nmemory table:", all_mem())

=== 1. plain answer (no tool) ===
TOOLS: (none)
MEDIA: (none)
MEMORY changed: (none)

REPLY:
An API (Application Programming Interface) is a set of defined rules and protocols that allows different software applications to communicate and interact with each other.

=== 2. run_python ===
TOOLS: ['run_python']
MEDIA: (none)
MEMORY changed: (none)

REPLY:
- 17 × 23 = **391**  
- Mean of \[4, 8, 15, 16, 23, 42\] = **18.0**

=== 3. web_search ===
TOOLS: ['web_search']
MEDIA: (none)
MEMORY changed: (none)

REPLY:
The current CEO of OpenAI is **Sam Altman**.

=== 4. generate_image ===
TOOLS: ['generate_image']
MEDIA: ['image']
MEMORY changed: (none)

REPLY:
Here’s a cartoon cat riding a bicycle:

![Cartoon cat riding a bicycle](attachment://generated_image.png)

=== 5. speak ===
TOOLS: ['speak']
MEDIA: ['audio']
MEMORY changed: (none)

REPLY:
I've spoken the phrase "hello from the mini ChatGPT" out loud for you.

=== 6. memory ===
TOOLS: (none)
MEDIA: (none)
MEMORY changed: [1, 2]

REPLY:
- E

In [11]:
print("\n=== 6. memory ==="); respond(cid, "I am not learning spanish, now i am learning kannada")
print("\nmemory table:", all_mem())


=== 6. memory ===
TOOLS: (none)
MEDIA: (none)
MEMORY changed: [1, 3]

REPLY:
- ಅರ್ಥವಾಯಿತು, ನೀವು ಈಗ ಕನ್ನಡವನ್ನು ಕಲಿಯುತ್ತಿದ್ದೀರಿ.  
- ಉತ್ತರಗಳನ್ನು ಕನ್ನಡದಲ್ಲಿ, ಸಂಕ್ಷಿಪ್ತ ಬುಲೆಟ್‌ ಪಾಯಿಂಟ್‌ಗಳಲ್ಲಿ ನೀಡುತ್ತೇನೆ.

memory table: [(2, 'preference', 'User prefers short bullet-point answers'), (3, 'fact', 'User is learning Kannada')]


## Step 8 - New chat, memory persists

Fresh conversation (no history). Two memory paths show at once: the **preference**
(bullet points) always applies, and `recall()` should surface the **fact** ("learning
Spanish") because it shares words with the question.

In [12]:
cid2 = new_conversation()
reply, trace, media, recalled, touched = respond(cid2,
    "Give me three tips for building Spanish vocabulary.")
print("\nrecalled this turn:", recalled)

TOOLS: (none)
MEDIA: (none)
MEMORY changed: (none)

REPLY:
- **Use spaced repetition (SRS):** Apps like Anki or Memrise schedule reviews right before you’re likely to forget a word, dramatically boosting long-term retention.
- **Learn words in context, not isolation:** Study phrases and short sentences (e.g., “tengo hambre” instead of just “hambre”) so you pick up natural usage, collocations, and grammar cues.
- **Consume content you enjoy:** Read graded readers, listen to podcasts, or watch shows with Spanish subtitles—repeated exposure to words in meaningful, enjoyable contexts cements them in your memory.

recalled this turn: []


## Step 9 - The app

- **left**: your chats (new / delete)
- **middle**: the conversation, a text box, a mic, and a file attach
- **right**: long-term memory + what was recalled / changed this turn, and any image/audio
  the last turn produced

Mic audio is transcribed by Whisper and dropped into the text box. The tool trace is
printed under each reply.

In [13]:
import gradio as gr

def mem_rows():
    return [[i, k, c] for i, k, c in all_mem()]

def refresh_list():
    convs = list_conversations()
    choices = [(t, i) for i, t in convs]
    return gr.update(choices=choices, value=(choices[0][1] if choices else None))

def load_conv(cid):
    return [{"role": r, "content": c} for r, c in get_messages(cid)] if cid else []

def on_audio(audio_path, textbox):
    if audio_path:
        return transcribe(audio_path)
    return textbox

def on_file(f):
    UPLOADED[0] = f.name if f else None
    return f"attached: {os.path.basename(UPLOADED[0])}" if UPLOADED[0] else "no file"

def send(cid, user_text, chat):
    if not user_text.strip():
        return chat, "", gr.update(), cid, mem_rows(), "(none)", "(none)", None, None
    if not cid:
        cid = new_conversation()
    reply, trace, media, recalled, touched = respond(cid, user_text, verbose=False)
    chat = load_conv(cid)
    if trace:
        chat.append({"role": "assistant", "content": f"_tools used: {', '.join(trace)}_"})
    img = next((p for k, p in media if k == "image"), None)
    aud = next((p for k, p in media if k == "audio"), None)
    rec = "\n".join(f"({k}) {c}" for _, k, c in recalled) or "(none)"
    chg = ", ".join(f"#{i}" for i in touched) or "(none)"
    return chat, "", refresh_list(), cid, mem_rows(), rec, chg, img, aud

def new_chat():
    cid = new_conversation()
    return cid, [], refresh_list()

def del_chat(cid):
    if cid:
        delete_conversation(cid)
    convs = list_conversations()
    ncid = convs[0][0] if convs else None
    return ncid, load_conv(ncid), refresh_list()

with gr.Blocks(title="GSSS_011 - Mini ChatGPT") as demo:
    gr.Markdown("# Mini ChatGPT - one model, many tools")
    cur = gr.State(None)
    with gr.Row():
        with gr.Column(scale=1):
            new_btn = gr.Button("New chat", variant="primary")
            chats = gr.Radio(label="Your chats", choices=[])
            del_btn = gr.Button("Delete chat", variant="stop")
            file_in = gr.File(label="Attach a file (csv / xlsx / txt)")
            file_note = gr.Markdown("no file")
        with gr.Column(scale=3):
            box = gr.Chatbot(height=430)
            msg = gr.Textbox(label="Message", placeholder="Type, or use the mic below")
            mic = gr.Audio(sources=["microphone"], type="filepath", label="Speak (Whisper -> text)")
        with gr.Column(scale=2):
            gr.Markdown("### Long-term memory")
            memtab = gr.Dataframe(headers=["id", "kind", "content"], interactive=False, wrap=True)
            recalled_box = gr.Textbox(label="recalled this turn", lines=2)
            changed_box = gr.Textbox(label="memory changed", lines=1)
            out_img = gr.Image(label="image out", height=220)
            out_aud = gr.Audio(label="audio out")

    demo.load(refresh_list, outputs=chats)
    demo.load(mem_rows, outputs=memtab)
    chats.change(lambda c: (c, load_conv(c)), chats, [cur, box])
    new_btn.click(new_chat, outputs=[cur, box, chats])
    del_btn.click(del_chat, cur, [cur, box, chats])
    file_in.change(on_file, file_in, file_note)
    mic.stop_recording(on_audio, [mic, msg], msg)
    msg.submit(send, [cur, msg, box],
               [box, msg, chats, cur, memtab, recalled_box, changed_box, out_img, out_aud])

demo.launch(debug=False)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9cd5f741d80b210a94.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Recap

Every ChatGPT-style feature here is a **small, separable part**:

- **history** and **memory** are just SQLite tables - one stores the transcript, one stores
  distilled facts the model curates.
- **tools** are plain Python functions; the model picks one from its **docstring**. Adding
  a capability = adding a function.
- **every external call has a fallback**: the agent model (qwen -> gpt-oss -> OpenRouter),
  `web_search` (DuckDuckGo -> Wikipedia), and `transcribe` (fails quietly so you can type).
- **voice in** is a pre-step (Whisper), **voice out** is a tool (gTTS). Small models
  sometimes *say* "I've read that aloud" without calling `speak`, so `respond()` keeps a
  deterministic safety net: if the user clearly asked for audio and the tool was skipped,
  it runs `speak` anyway.
- the **agent** is the glue: recall memory -> build the system prompt -> let the model run
  tools -> persist -> reconcile memory.

### Exercises
1. Stream the reply: after the agent finishes, yield the text word by word into the Chatbot.
2. Store which tools each message used in a new `messages.tools` column; show it in the sidebar.
3. Swap the agent model to `openai/gpt-oss-20b` and see how many tool calls it gets wrong.
4. Add a "voice mode": if the incoming message came from the mic, automatically `speak` the reply.
5. Swap TF-IDF recall back to MiniLM embeddings and compare - does "what's for dinner?" now
   recall "I'm vegetarian" even though they share no words?
